In [1]:
import numpy as np
import pandas as pd
import os
import PIL
from PIL import Image 

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

### Dataset

In [2]:
img_path = "/home/rahul/code/gsoc/Evaluation-Test-ArtExtract/Task1/wikiart"


def filter_existing_files(df, root_dir):
    # This checks every single row to see if the file is actually there
    exists = df['filepath'].apply(lambda x: os.path.exists(os.path.join(root_dir, x)))
    initial_count = len(df)
    df = df[exists]
    print(f"Kept {len(df)} images. Dropped {initial_count - len(df)} missing files.")
    return df

# Train dataset
style_path_train = "/home/rahul/code/gsoc/Evaluation-Test-ArtExtract/Task1/wikiart_csv/style_train.csv"
genre_path_train = "/home/rahul/code/gsoc/Evaluation-Test-ArtExtract/Task1/wikiart_csv/genre_train.csv"
artist_path_train = "/home/rahul/code/gsoc/Evaluation-Test-ArtExtract/Task1/wikiart_csv/artist_train.csv"

artist_df = pd.read_csv(artist_path_train,header=None,names=['filepath','artist_label'])
genre_df = pd.read_csv(genre_path_train,header=None,names=['filepath','genre_label'])
style_df = pd.read_csv(style_path_train,header=None,names=['filepath','style_label'])

merged_df_temp_train = pd.merge(artist_df,style_df)
merged_df_train = pd.merge(merged_df_temp_train,genre_df)
merged_df_train = filter_existing_files(merged_df_train, img_path)



# Validation dataset
style_path_val = "/home/rahul/code/gsoc/Evaluation-Test-ArtExtract/Task1/wikiart_csv/style_val.csv"
genre_path_val = "/home/rahul/code/gsoc/Evaluation-Test-ArtExtract/Task1/wikiart_csv/genre_val.csv"
artist_path_val = "/home/rahul/code/gsoc/Evaluation-Test-ArtExtract/Task1/wikiart_csv/artist_val.csv"

artist_df = pd.read_csv(artist_path_val,header=None,names=['filepath','artist_label'])
genre_df = pd.read_csv(genre_path_val,header=None,names=['filepath','genre_label'])
style_df = pd.read_csv(style_path_val,header=None,names=['filepath','style_label'])

merged_df_temp_val = pd.merge(artist_df,style_df)
merged_df_val = pd.merge(merged_df_temp_val,genre_df)
merged_df_val = filter_existing_files(merged_df_val, img_path)

Kept 11274 images. Dropped 2 missing files.
Kept 4707 images. Dropped 0 missing files.


In [3]:
class ArtDataset(torch.utils.data.Dataset):
    def __init__(self,merged_df_all,img_path,transform=None):
        self.merged_df = merged_df_all
        self.img_path = img_path
        self.transform=transform

    def __len__(self):
        return len(self.merged_df)
    
    def __getitem__(self,id):
        row = self.merged_df.iloc[id]
        path = row['filepath']
        artist_idx = row['artist_label']
        genre_idx = row['genre_label']
        style_idx = row['style_label']
        
        full_path = os.path.join(self.img_path,path)
        img = Image.open(full_path).convert('RGB')
        if self.transform:
            img = self.transform(img)

        return img,(artist_idx,genre_idx,style_idx),full_path    


train_transforms = torchvision.transforms.Compose(
    [
        torchvision.transforms.Resize(512),
        torchvision.transforms.RandomCrop(224),
        # RandomHorizontalFlip to prevent overfitting
        torchvision.transforms.RandomHorizontalFlip(p=0.5),
        torchvision.transforms.ToTensor(),
        # standard ImageNet normalization
        torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]
)

val_transforms = torchvision.transforms.Compose(
    [
        torchvision.transforms.Resize(512),
        torchvision.transforms.RandomCrop(224),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]
)

### Model

In [4]:
class ResidualBlock(nn.Module):
    def __init__(self, inchannel, outchannel, stride=1):
        super(ResidualBlock, self).__init__()
        self.left = nn.Sequential(
            nn.Conv2d(inchannel, outchannel, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(outchannel),
            nn.ReLU(inplace=True),
            nn.Conv2d(outchannel, outchannel, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(outchannel)
        )
        self.shortcut = nn.Sequential()
        if stride != 1 or inchannel != outchannel:
            self.shortcut = nn.Sequential(
                nn.Conv2d(inchannel, outchannel, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(outchannel)
            )
            
    def forward(self, x):
        out = self.left(x)
        out = out + self.shortcut(x)
        out = F.relu(out)
        
        return out

class ResNet(nn.Module):
    def __init__(self, ResidualBlock):
        super(ResNet, self).__init__()
        self.inchannel = 64
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.layer1 = self.make_layer(ResidualBlock, 64, 2, stride=1)
        self.layer2 = self.make_layer(ResidualBlock, 128, 2, stride=2)
        self.layer3 = self.make_layer(ResidualBlock, 256, 2, stride=2)        
        self.layer4 = self.make_layer(ResidualBlock, 512, 2, stride=2)        
        
    def make_layer(self, block, channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for stride in strides:
            layers.append(block(self.inchannel, channels, stride))
            self.inchannel = channels
        return nn.Sequential(*layers)
    
    def forward(self, x):
        out = self.conv1(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        return out


class BiLSTM(torch.nn.Module):
    def __init__(self,input_dim,hidden_dim,num_layers,num_styles,num_artists,num_genres,batch_first=True):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.num_styles = num_styles
        self.num_artists = num_artists
        self.num_genres = num_genres

        self.num_directions = 2

        self.lstm = torch.nn.LSTM(
            input_size = input_dim,
            hidden_size = hidden_dim,
            num_layers = num_layers,
            bidirectional = True,
            batch_first = batch_first,
        )

        self.fc_artist= torch.nn.Linear(hidden_dim*self.num_directions,self.num_artists)
        self.fc_genre = torch.nn.Linear(hidden_dim*self.num_directions,self.num_genres)
        self.fc_style = torch.nn.Linear(hidden_dim*self.num_directions,self.num_styles)

    
    def forward(self,x):
        h0 = torch.zeros(self.num_layers*self.num_directions,x.size(0),self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers*self.num_directions,x.size(0),self.hidden_dim).to(x.device)

        output,(hn,cn) = self.lstm(x,(h0,c0))

        hn = hn.view(self.num_layers,self.num_directions,x.size(0),self.hidden_dim)

        final_hidden_state = torch.cat((hn[-1,0,:,:],hn[-1,1,:,:]),dim=1)

        out1 = self.fc_artist(final_hidden_state)
        out2 = self.fc_genre(final_hidden_state)
        out3 = self.fc_style(final_hidden_state)

        return (out1,out2,out3)



class ArtClassifier_CRNN(torch.nn.Module):
    def __init__(self,hidden_dim,num_layers,num_styles,num_artists,num_genres,):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.num_styles = num_styles
        self.num_artists = num_artists
        self.num_genres = num_genres

        self.ResNet = ResNet(ResidualBlock)
        self.BiLSTM = BiLSTM(512,hidden_dim=self.hidden_dim,num_layers=self.num_layers,num_styles=self.num_styles,num_artists=self.num_artists,num_genres=self.num_genres)
    
    def forward(self,input):
        features = self.ResNet(input)
        features = features.flatten(start_dim=2).permute(0,2,1)
        out = self.BiLSTM(features)

        return out

### Training Loop

In [5]:
from sklearn.metrics import f1_score

merged_df_train['artist_label'], artist_uniques = pd.factorize(merged_df_train['artist_label'])
merged_df_train['genre_label'], genre_uniques = pd.factorize(merged_df_train['genre_label'])
merged_df_train['style_label'], style_uniques = pd.factorize(merged_df_train['style_label'])

# dictionaries for mapping (String -> Integer)
artist_map = {name: i for i, name in enumerate(artist_uniques)}
genre_map = {name: i for i, name in enumerate(genre_uniques)}
style_map = {name: i for i, name in enumerate(style_uniques)}

merged_df_val['artist_label'] = merged_df_val['artist_label'].map(artist_map)
merged_df_val['genre_label'] = merged_df_val['genre_label'].map(genre_map)
merged_df_val['style_label'] = merged_df_val['style_label'].map(style_map)

merged_df_val = merged_df_val.dropna()
merged_df_val = merged_df_val.astype({'artist_label': 'int64', 'genre_label': 'int64', 'style_label': 'int64'})

model = ArtClassifier_CRNN(hidden_dim=512,num_artists=len(artist_uniques),num_genres=len(genre_uniques),num_styles=len(style_uniques),num_layers=2).to('cuda')
optimizer = torch.optim.Adam(params=model.parameters(),lr=1e-4)
criterion = torch.nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

dataset = ArtDataset(merged_df_train,img_path,transform=train_transforms)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)

val_dataset = ArtDataset(merged_df_val,img_path,transform=val_transforms)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=True)

num_epoch = 50
best_avg_f1=0
outliers = []

for epoch in range(num_epoch):
    model.train()
    
    for i,batch in enumerate(dataloader):
        img,(aidx,gidx,sidx),paths = batch
        img = img.to('cuda')
        aidx = aidx.to('cuda')
        gidx = gidx.to('cuda')
        sidx = sidx.to('cuda')

        optimizer.zero_grad()

        out_artist, out_genre, out_style = model(img)

        loss_artist = criterion(out_artist,aidx)
        loss_genre = criterion(out_genre,gidx)
        loss_style = criterion(out_style,sidx)
        total_loss = loss_artist+loss_genre+loss_style

        total_loss.backward()
        optimizer.step()

        if i%100==0:
            print(f"Epoch [{epoch+1}/{num_epoch}] Batch [{i}] Loss: {total_loss.item():.4f}")
    
    model.eval()
    all_artist_preds = []
    all_artist_targets = []

    all_genre_preds = []
    all_genre_targets = []

    all_style_preds = []
    all_style_targets = []

    with torch.no_grad():
        for val_batch in val_dataloader:
            img,(aidx,gidx,sidx),paths = val_batch
            img = img.to('cuda')
            aidx = aidx.to('cuda')
            gidx = gidx.to('cuda')
            sidx = sidx.to('cuda')

            out_artist, out_genre, out_style = model(img)

            artist_preds = torch.argmax(out_artist,dim=1)
            genre_preds = torch.argmax(out_genre,dim=1)
            style_preds = torch.argmax(out_style,dim=1)

            all_artist_preds.extend(artist_preds.cpu().numpy())
            all_genre_preds.extend(genre_preds.cpu().numpy())
            all_style_preds.extend(style_preds.cpu().numpy())

            all_artist_targets.extend(aidx.cpu().numpy())
            all_genre_targets.extend(gidx.cpu().numpy())
            all_style_targets.extend(sidx.cpu().numpy())

            # outlier logic
            probs = F.softmax(out_artist,dim=1)
            confidences, preds = torch.max(probs,dim=1)
            for i in range(len(preds)):
                if preds[i] != aidx[i] and confidences[i]>0.95: 
                    true_name = artist_uniques[aidx[i].item()] 
                    pred_name = artist_uniques[preds[i].item()]
                    outliers.append(f"Image: {paths[i]} | True: {true_name} | Predicted: {pred_name} | Conf: {confidences[i]:.4f}")
    

    a_macro_f1 = f1_score(all_artist_targets,all_artist_preds,average='macro')
    g_macro_f1 = f1_score(all_genre_targets,all_genre_preds,average='macro')
    s_macro_f1 = f1_score(all_style_targets,all_style_preds,average='macro')
    print(f"\nEpoch {epoch+1}:\nArtist Macro F1: {a_macro_f1}\nGenre Macro F1: {g_macro_f1}\nStyle Macro F1: {s_macro_f1}\n")
    print("\n\n")
    

    current_avg_f1 = (a_macro_f1 + g_macro_f1 + s_macro_f1) / 3
    scheduler.step(current_avg_f1)

    if current_avg_f1 > best_avg_f1:
        best_avg_f1 = current_avg_f1
        torch.save(model.state_dict(), 'best_art_model.pth')
        print(f"--- Model Saved! New Best Avg F1: {best_avg_f1:.4f} ---")

    

Epoch [1/50] Batch [0] Loss: 8.1886
Epoch [1/50] Batch [100] Loss: 6.7817
Epoch [1/50] Batch [200] Loss: 6.0424
Epoch [1/50] Batch [300] Loss: 7.1816

Epoch 1:
Artist Macro F1: 0.11210809795937281
Genre Macro F1: 0.22267869571227497
Style Macro F1: 0.13942511159207238




--- Model Saved! New Best Avg F1: 0.1581 ---
Epoch [2/50] Batch [0] Loss: 6.5182
Epoch [2/50] Batch [100] Loss: 6.3496
Epoch [2/50] Batch [200] Loss: 5.7687
Epoch [2/50] Batch [300] Loss: 5.6417

Epoch 2:
Artist Macro F1: 0.16032828168473126
Genre Macro F1: 0.23496575151400875
Style Macro F1: 0.16127158058952165




--- Model Saved! New Best Avg F1: 0.1855 ---
Epoch [3/50] Batch [0] Loss: 6.0649
Epoch [3/50] Batch [100] Loss: 5.9984
Epoch [3/50] Batch [200] Loss: 5.6784
Epoch [3/50] Batch [300] Loss: 5.8221

Epoch 3:
Artist Macro F1: 0.15697956810771888
Genre Macro F1: 0.24433174259094298
Style Macro F1: 0.15532905130036462




--- Model Saved! New Best Avg F1: 0.1855 ---
Epoch [4/50] Batch [0] Loss: 5.5147
Epoch [4/5